In [1]:
import pandas as pd
from openai import OpenAI
from data_loader import BiosDataset, SBICDataset

import numpy as np
import os
import pickle
import yaml
import json
from tqdm import tqdm

## Copy SBIC embeddings from full dataset to avoid computing them twice

In [2]:
MODEL_PLACEHOLDER = 'MODEL'
SPLIT_PLACEHOLDER = 'SPLIT'
DATASET_PLACEHOLDER = 'DATASET'
models = ['text-embedding-3-small', 'text-embedding-3-large']
split = 'test'
emb_file = '../../artifacts/embeddings/DATASET_SPLIT_MODEL.pickle'

prefix = '../../data/sbic_reannot/' # adapt data if stored elsewhere

local_paths = {
    'data': prefix+'review_full.csv',
    'label_map_orig': prefix+'freetext_label_lookup_original.csv',
    'label_hierarchy': prefix+'label_hierarchy.csv',
    'label_category': prefix+'label_categories.csv'
}

In [3]:
full = SBICDataset(local_dir=local_paths, option='full')
subset = SBICDataset(local_dir=local_paths, option='subset-orig')

full.name = 'sbic-full'
subset.name = 'sbic-sorig'

load SBIC with option full and local file: {'data': '../../data/sbic_reannot/review_full.csv', 'label_map_orig': '../../data/sbic_reannot/freetext_label_lookup_original.csv', 'label_hierarchy': '../../data/sbic_reannot/label_hierarchy.csv', 'label_category': '../../data/sbic_reannot/label_categories.csv'}
compute class weights for split train
compute class weights for split test
compute class weights for split dev
load SBIC with option subset-orig and local file: {'data': '../../data/sbic_reannot/review_full.csv', 'label_map_orig': '../../data/sbic_reannot/freetext_label_lookup_original.csv', 'label_hierarchy': '../../data/sbic_reannot/label_hierarchy.csv', 'label_category': '../../data/sbic_reannot/label_categories.csv'}
compute class weights for split test


In [9]:
# create mapping from subset to full
df_full = pd.DataFrame({'index': list(np.arange(len(full.data[split]))), 'HITId': full.ids[split], 'post': full.data[split]})
df_subset = pd.DataFrame({'index_s': list(np.arange(len(subset.data[split]))), 'HITId': subset.ids[split], 'post': subset.data[split]})

mg = pd.merge(df_subset, df_full, on='HITId')#.drop_duplicates(subset=['index_s'], keep='first')
assert list(set(mg['index_s'])) == list(mg['index_s'])

mg

,index_s,HITId,post_x,index,post_y
0,0,301KG0KX9CF8K5JONRZNDVHCQ67H2B,4) safety gloves. It's time we start calling o...,2,4) safety gloves. It's time we start calling o...
1,1,301KG0KX9CF8K5JONRZNDVHCS2P2H8,well since we have all of us from phoenix lets...,6,well since we have all of us from phoenix lets...
2,2,302OLP89DZ16803CEEE18M3UQ0GACL,Disclaimer: None of these things I'm about to ...,11,Disclaimer: None of these things I'm about to ...
3,3,302OLP89DZ16803CEEE18M3UR0RCAZ,Boycott: Popular Washington DC area Radio Stat...,13,Boycott: Popular Washington DC area Radio Stat...
4,4,302OLP89DZ16803CEEE18M3UR0TCA1,Why does the myth that women rarely lie about ...,14,Why does the myth that women rarely lie about ...
...,...,...,...,...,...
1537,1537,3ZZAYRN1I6LCWLZOAJE9YRN31SATOO,"What's red, white, and black ?",4696,"What's red, white, and black ?"
1538,1538,3ZZAYRN1I6LCWLZOAJE9YRN31WSOT9,What's worse than a field-trip to an art Museu...,4697,What's worse than a field-trip to an art Museu...
1539,1539,3ZZAYRN1I6LCWLZOAJE9YRN3254OT4,You know a bitch mad when she throw her drink ...,4699,You know a bitch mad when she throw her drink ...
1540,1540,3ZZAYRN1I6LCWLZOAJE9YRN325YTO3,your not gonna change a nigga.he is who he is ...,4700,your not gonna change a nigga.he is who he is ...


In [12]:
split = 'test'
for model in models:
    # get embeddings from full dataset
    emb_file_ = emb_file.replace(DATASET_PLACEHOLDER, full.name).replace(MODEL_PLACEHOLDER, model).replace(SPLIT_PLACEHOLDER, split)
    with open(emb_file_, 'rb') as handle:
        content = pickle.load(handle)
        print(content.keys())
    emb = content['embeddings']

    assert content['model'] == model
    assert emb.shape[0] == len(full.data[split])

    # get subset indices and embeddings
    indices = list(mg['index'])
    emb_sub = emb[indices,:]

    assert emb_sub.shape[0] == len(subset.data[split]), f"{emb_sub.shape} / {len(subset.data[split])}"

    # use short form of dataset name because all subsets share text / embeddings
    save_file = emb_file.replace(DATASET_PLACEHOLDER, 'sbic-s').replace(MODEL_PLACEHOLDER, model).replace(SPLIT_PLACEHOLDER, split)
    save_dict = {'model': model, 'embeddings': emb_sub}
    with open(save_file, 'wb') as handle:
        pickle.dump(save_dict, handle)

dict_keys(['model', 'embeddings'])
../../artifacts/embeddings/sbic-s_test_text-embedding-3-small.pickle
dict_keys(['model', 'embeddings'])
../../artifacts/embeddings/sbic-s_test_text-embedding-3-large.pickle


## Copy BIOS embeddings from full dataset to avoid computing them twice

In [48]:
MODEL_PLACEHOLDER = 'MODEL'
SPLIT_PLACEHOLDER = 'SPLIT'
DATASET_PLACEHOLDER = 'DATASET'
models = ['text-embedding-3-small', 'text-embedding-3-large']
splits = ['train', 'dev', 'test']
emb_file = '../../artifacts/embeddings/DATASET_SPLIT_MODEL.pickle'

In [54]:
def load_dataset(name: str, option: str, local_dir: str = None):
    dataset = BiosDataset(local_dir=local_dir, option=option)
    dataset.name = name
    return dataset

bios_s = load_dataset(name='bios-supervised', option='supervised', local_dir='../../data/bios_huggingface_merge.pkl')
bios_s2 = load_dataset(name='bios-supervised-v2', option='supervised-v2', local_dir='../../data/biosbias_review_artifacts/huggingface_merge_complete_v2.pkl')
bios = load_dataset(name='bios-unsupervised', option='unsupervised', local_dir=None)

load BIOS with option supervised
compute class weights for split train
compute class weights for split test
compute class weights for split dev
load BIOS with option supervised-v2
compute class weights for split train
compute class weights for split test
compute class weights for split dev
load BIOS with option unsupervised
compute class weights for split train
compute class weights for split test
compute class weights for split dev


In [ ]:
# check which embeddings exist on disk
for model in models:
    print(model)
    for dataset in [bios_s, bios_s2, bios]:
        print(dataset.name)
        for split in splits:
            emb_file_ = emb_file.replace(DATASET_PLACEHOLDER, dataset.name).replace(MODEL_PLACEHOLDER, model).replace(SPLIT_PLACEHOLDER, split)
            found = os.path.isfile(emb_file_)
            if found:
                print("found ", emb_file_)
                with open(emb_file_, 'rb') as handle:
                    content = pickle.load(handle)
                print("emb/content match: ", content['embeddings'].shape[0] == len(dataset.data[split]))
                #print(content['embeddings'].shape)
                #print(len(dataset.data[split]))
            else:
                print("could not find ", emb_file_)

        print()
        print()

In [75]:
for model in models:
    for split in splits:
        # get embeddings from full dataset
        emb_file_ = emb_file.replace(DATASET_PLACEHOLDER, bios.name).replace(MODEL_PLACEHOLDER, model).replace(SPLIT_PLACEHOLDER, split)
        with open(emb_file_, 'rb') as handle:
            content = pickle.load(handle)
            print(content.keys())
        emb = content['embeddings']

        assert content['model'] == model
        assert emb.shape[0] == len(bios.data[split])

        # merge supervised v2 version and full dataset on texts to get embedding indices
        df = pd.DataFrame({'index': list(np.arange(len(bios.data[split]))), 'data': bios.data[split]})
        df_s = pd.DataFrame({'index_s': list(np.arange(len(bios_s2.data[split]))), 'data': bios_s2.data[split]})
        mg = pd.merge(df_s, df, on='data').drop_duplicates(subset=['index_s'], keep='first')

        assert list(set(mg['index_s'])) == list(mg['index_s'])

        indices = list(mg['index'])
        emb_s2 = emb[indices,:]

        assert emb_s2.shape[0] == len(bios_s2.data[split]), f"{emb_s2.shape} / {len(bios_s2.data[split])}"

        save_file = emb_file.replace(DATASET_PLACEHOLDER, bios_s2.name).replace(MODEL_PLACEHOLDER, model).replace(SPLIT_PLACEHOLDER, split)
        save_dict = {'model': model, 'embeddings': emb_s2}
        with open(save_file, 'wb') as handle:
            pickle.dump(save_dict, handle)

dict_keys(['model', 'embeddings'])
dict_keys(['model', 'embeddings'])
dict_keys(['model', 'embeddings'])
dict_keys(['model', 'embeddings'])
dict_keys(['model', 'embeddings'])
dict_keys(['model', 'embeddings'])


In [76]:
from pie_data import get_dataset

dataset_name = 'bios-unsupervised'
X_train, y_train, X_test, y_test, n_classes, multi_label, class_weights, protected_attr_dict = get_dataset(dataset_name)
X_test == bios.data['test']

False